In [1]:
import pandas as pd
import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.metrics import mean_squared_error
import warnings

warnings.filterwarnings("ignore")
import os

# Set plotting style
sns.set_theme(style="whitegrid")

# Define local paths
local_dir = os.path.dirname(os.path.abspath("__file__")) or os.getcwd()
output_dir = os.path.join(local_dir, "output")
os.makedirs(output_dir, exist_ok=True)

In [2]:
df_train = pd.read_csv("bike_train.csv")
df_test = pd.read_csv("bike_test.csv")

In [3]:
print(f"Training set shape: {df_train.shape}")
print(f"Test set shape: {df_test.shape}")
print(f"\nMissing values in train: {df_train.isna().sum().sum()}")
print(f"Missing values in test: {df_test.isna().sum().sum()}")

print("\n=== Training Data Info ===")
print(df_train.info())
print(df_train.head())
print("\n")
print(df_train.describe())

# Check data types
print("\nData types:")
print(df_train.dtypes)

Training set shape: (10450, 12)
Test set shape: (2613, 9)

Missing values in train: 0
Missing values in test: 0

=== Training Data Info ===
<class 'pandas.DataFrame'>
RangeIndex: 10450 entries, 0 to 10449
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   datetime    10450 non-null  str    
 1   season      10450 non-null  int64  
 2   holiday     10450 non-null  int64  
 3   workingday  10450 non-null  int64  
 4   weather     10450 non-null  int64  
 5   temp        10450 non-null  float64
 6   atemp       10450 non-null  float64
 7   humidity    10450 non-null  int64  
 8   windspeed   10450 non-null  float64
 9   casual      10450 non-null  int64  
 10  registered  10450 non-null  int64  
 11  count       10450 non-null  int64  
dtypes: float64(3), int64(8), str(1)
memory usage: 979.8 KB
None
              datetime  season  holiday  workingday  weather      temp  \
0   2012-07-15 7:00:00       3        0          

In [5]:
for col in ["season", "holiday", "workingday", "weather"]:
    print(df_train[col].value_counts())

season
3    2630
4    2630
2    2608
1    2582
Name: count, dtype: int64
holiday
0    10149
1      301
Name: count, dtype: int64
workingday
1    7061
0    3389
Name: count, dtype: int64
weather
1    6945
2    2686
3     818
4       1
Name: count, dtype: int64


In [ ]:
def preprocess_and_engineer(df):
    """Advanced feature engineering for bike demand prediction"""
    df_copy = df.copy()

    # Parse datetime with mixed format handling
    df_copy["datetime_dt"] = pd.to_datetime(
        df_copy["datetime"], format="mixed", dayfirst=False
    )

    # Extract temporal features
    df_copy["hour"] = df_copy["datetime_dt"].dt.hour
    df_copy["month"] = df_copy["datetime_dt"].dt.month
    df_copy["year"] = df_copy["datetime_dt"].dt.year - 2011  # 0 for 2011, 1 for 2012
    df_copy["day_of_week"] = df_copy["datetime_dt"].dt.dayofweek
    df_copy["day"] = df_copy["datetime_dt"].dt.day

    # Cyclical encoding for hours (sine/cosine for circular pattern)
    df_copy["hour_sin"] = np.sin(2 * np.pi * df_copy["hour"] / 24)
    df_copy["hour_cos"] = np.cos(2 * np.pi * df_copy["hour"] / 24)
    df_copy["month_sin"] = np.sin(2 * np.pi * df_copy["month"] / 12)
    df_copy["month_cos"] = np.cos(2 * np.pi * df_copy["month"] / 12)
    df_copy["dayofweek_sin"] = np.sin(2 * np.pi * df_copy["day_of_week"] / 7)
    df_copy["dayofweek_cos"] = np.cos(2 * np.pi * df_copy["day_of_week"] / 7)

    # Peak hour indicators
    df_copy["is_peak_hour"] = (
        ((df_copy["hour"] == 8) | (df_copy["hour"] == 17) | (df_copy["hour"] == 18))
        & (df_copy["workingday"] == 1)
    ).astype(int)

    df_copy["is_night"] = ((df_copy["hour"] >= 22) | (df_copy["hour"] <= 5)).astype(int)
    df_copy["is_afternoon"] = (
        (df_copy["hour"] >= 12) & (df_copy["hour"] <= 16)
    ).astype(int)

    # Weather interactions
    df_copy["temp_humidity"] = df_copy["temp"] * df_copy["humidity"]
    df_copy["temp_atemp"] = df_copy["temp"] * df_copy["atemp"]
    df_copy["temp_windspeed"] = df_copy["temp"] * df_copy["windspeed"]
    df_copy["humidity_windspeed"] = df_copy["humidity"] * df_copy["windspeed"]

    # Polynomial weather features
    df_copy["temp_sq"] = df_copy["temp"] ** 2
    df_copy["humidity_sq"] = df_copy["humidity"] ** 2
    df_copy["windspeed_sq"] = df_copy["windspeed"] ** 2

    # Drop the temporary datetime column
    df_copy = df_copy.drop(columns=["datetime_dt"])

    return df_copy


df_train_eng = preprocess_and_engineer(df_train)
df_test_eng = preprocess_and_engineer(df_test)

print(
    f"\nEngineered features created. Train shape: {df_train_eng.shape}, Test shape: {df_test_eng.shape}"
)
print(f"\nNew feature columns: {list(df_train_eng.columns)}")

In [ ]:
# Define RMSLE metric
def get_rmsle(y_true, y_pred):
    """Calculate Root Mean Squared Logarithmic Error"""
    y_pred = np.clip(y_pred, 0, None)  # Ensure non-negative predictions
    return np.sqrt(np.mean((np.log1p(y_pred) - np.log1p(y_true)) ** 2))


# Prepare data
X = df_train_eng.drop(columns=["datetime", "count"], errors="ignore")
y = df_train_eng["count"]

# Train-validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"Training set size: {X_train.shape[0]}")
print(f"Validation set size: {X_val.shape[0]}")

# ============== MODEL 1: Baseline ==============
print("\n" + "=" * 60)
print("MODEL 1: Baseline (Simple weather features only)")
print("=" * 60)

base_cols = ["temp", "humidity", "windspeed"]
lr_base = LinearRegression()
lr_base.fit(X_train[base_cols], y_train)
pred_tr_base = np.clip(lr_base.predict(X_train[base_cols]), 0, None)
pred_val_base = np.clip(lr_base.predict(X_val[base_cols]), 0, None)

train_rmsle_base = get_rmsle(y_train, pred_tr_base)
val_rmsle_base = get_rmsle(y_val, pred_val_base)

print(f"Train RMSLE: {train_rmsle_base:.6f}")
print(f"Validation RMSLE: {val_rmsle_base:.6f}")

# ============== MODEL 2: Engineered Features with OLS ==============
print("\n" + "=" * 60)
print("MODEL 2: Engineered Features (OLS on log-transformed target)")
print("=" * 60)

lr_eng = LinearRegression()
lr_eng.fit(X_train, np.log1p(y_train))
pred_tr_eng = np.expm1(lr_eng.predict(X_train))
pred_val_eng = np.expm1(lr_eng.predict(X_val))
pred_tr_eng = np.clip(pred_tr_eng, 0, None)
pred_val_eng = np.clip(pred_val_eng, 0, None)

train_rmsle_eng = get_rmsle(y_train, pred_tr_eng)
val_rmsle_eng = get_rmsle(y_val, pred_val_eng)

print(f"Train RMSLE: {train_rmsle_eng:.6f}")
print(f"Validation RMSLE: {val_rmsle_eng:.6f}")

In [ ]:
# ============== MODEL 3 & 4: Polynomial Features with Ridge & Lasso ==============
print("\n" + "=" * 60)
print("MODEL 3 & 4: Polynomial Features (degree=2) with Regularization")
print("=" * 60)

# Create polynomial features on weather columns
weather_cols = ["temp", "humidity", "windspeed"]
poly = PolynomialFeatures(degree=2, include_bias=False)

poly_tr = poly.fit_transform(X_train[weather_cols])
poly_val = poly.transform(X_val[weather_cols])

# Combine polynomial weather features with other engineered features
poly_tr_df = pd.DataFrame(
    poly_tr, columns=poly.get_feature_names_out(weather_cols), index=X_train.index
)
poly_val_df = pd.DataFrame(
    poly_val, columns=poly.get_feature_names_out(weather_cols), index=X_val.index
)

# Drop original weather columns and combine with polynomial
X_train_poly = pd.concat([X_train.drop(columns=weather_cols), poly_tr_df], axis=1)
X_val_poly = pd.concat([X_val.drop(columns=weather_cols), poly_val_df], axis=1)

# Scale features
scaler = StandardScaler()
X_train_poly_sc = scaler.fit_transform(X_train_poly)
X_val_poly_sc = scaler.transform(X_val_poly)

print(f"Polynomial feature space shape: {X_train_poly_sc.shape}")

# Ridge Tuning
print("\nTuning Ridge Regression...")
best_ridge_alpha = None
best_ridge_val_rmsle = float("inf")
ridge_results = {}

for alpha in [0.001, 0.01, 0.1, 0.5, 1.0, 5.0, 10.0, 50.0, 100.0]:
    ridge = Ridge(alpha=alpha, max_iter=5000)
    ridge.fit(X_train_poly_sc, np.log1p(y_train))
    pred_val = np.clip(np.expm1(ridge.predict(X_val_poly_sc)), 0, None)
    val_rmsle = get_rmsle(y_val, pred_val)
    ridge_results[alpha] = val_rmsle

    if val_rmsle < best_ridge_val_rmsle:
        best_ridge_val_rmsle = val_rmsle
        best_ridge_alpha = alpha

print(
    f"Best Ridge alpha: {best_ridge_alpha} with validation RMSLE: {best_ridge_val_rmsle:.6f}"
)

# Train final Ridge model
ridge_final = Ridge(alpha=best_ridge_alpha, max_iter=5000)
ridge_final.fit(X_train_poly_sc, np.log1p(y_train))
pred_tr_ridge = np.clip(np.expm1(ridge_final.predict(X_train_poly_sc)), 0, None)
pred_val_ridge = np.clip(np.expm1(ridge_final.predict(X_val_poly_sc)), 0, None)

train_rmsle_ridge = get_rmsle(y_train, pred_tr_ridge)
print(
    f"Ridge - Train RMSLE: {train_rmsle_ridge:.6f}, Val RMSLE: {best_ridge_val_rmsle:.6f}"
)

In [ ]:
# Lasso Tuning
print("\nTuning Lasso Regression...")
best_lasso_alpha = None
best_lasso_val_rmsle = float("inf")
lasso_results = {}

for alpha in [0.00001, 0.0001, 0.0005, 0.001, 0.005, 0.01, 0.05, 0.1, 0.5]:
    lasso = Lasso(alpha=alpha, max_iter=10000, warm_start=False)
    lasso.fit(X_train_poly_sc, np.log1p(y_train))
    pred_val = np.clip(np.expm1(lasso.predict(X_val_poly_sc)), 0, None)
    val_rmsle = get_rmsle(y_val, pred_val)
    lasso_results[alpha] = val_rmsle

    if val_rmsle < best_lasso_val_rmsle:
        best_lasso_val_rmsle = val_rmsle
        best_lasso_alpha = alpha

print(
    f"Best Lasso alpha: {best_lasso_alpha} with validation RMSLE: {best_lasso_val_rmsle:.6f}"
)

# Train final Lasso model
lasso_final = Lasso(alpha=best_lasso_alpha, max_iter=10000, warm_start=False)
lasso_final.fit(X_train_poly_sc, np.log1p(y_train))
pred_tr_lasso = np.clip(np.expm1(lasso_final.predict(X_train_poly_sc)), 0, None)
pred_val_lasso = np.clip(np.expm1(lasso_final.predict(X_val_poly_sc)), 0, None)

train_rmsle_lasso = get_rmsle(y_train, pred_tr_lasso)
print(
    f"Lasso - Train RMSLE: {train_rmsle_lasso:.6f}, Val RMSLE: {best_lasso_val_rmsle:.6f}"
)

In [ ]:
# Compile results
results_summary = pd.DataFrame(
    [
        {
            "Model": "Baseline OLS",
            "Train RMSLE": train_rmsle_base,
            "Val RMSLE": val_rmsle_base,
            "Features": "temp, humidity, windspeed",
        },
        {
            "Model": "Engineered OLS",
            "Train RMSLE": train_rmsle_eng,
            "Val RMSLE": val_rmsle_eng,
            "Features": "Engineered temporal + weather",
        },
        {
            "Model": f"Ridge (α={best_ridge_alpha})",
            "Train RMSLE": train_rmsle_ridge,
            "Val RMSLE": best_ridge_val_rmsle,
            "Features": "Polynomial + temporal",
        },
        {
            "Model": f"Lasso (α={best_lasso_alpha})",
            "Train RMSLE": train_rmsle_lasso,
            "Val RMSLE": best_lasso_val_rmsle,
            "Features": "Polynomial + temporal",
        },
    ]
)

print("\n" + "=" * 80)
print("MODEL COMPARISON")
print("=" * 80)
print(results_summary.to_string(index=False))

# Select best model
best_model_idx = results_summary["Val RMSLE"].idxmin()
best_model_name = results_summary.loc[best_model_idx, "Model"]
best_val_rmsle = results_summary.loc[best_model_idx, "Val RMSLE"]

print(f"\n✓ BEST MODEL: {best_model_name}")
print(f"  Validation RMSLE: {best_val_rmsle:.6f}")

In [ ]:
# Use best model (Lasso) for residual analysis
residuals = y_val - pred_val_lasso

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Residuals vs Predicted
axes[0].scatter(pred_val_lasso, residuals, alpha=0.5, color="darkviolet", s=30)
axes[0].axhline(y=0, color="red", linestyle="--", linewidth=2)
axes[0].set_title(
    "Residuals vs. Predicted Values (Lasso Model)", fontsize=12, fontweight="bold"
)
axes[0].set_xlabel("Predicted Counts")
axes[0].set_ylabel("Residuals")

# Residual Distribution
axes[1].hist(residuals, bins=30, edgecolor="black", alpha=0.7, color="teal")
axes[1].axvline(x=0, color="red", linestyle="--", linewidth=2)
axes[1].set_title("Distribution of Residuals", fontsize=12, fontweight="bold")
axes[1].set_xlabel("Residual Value")
axes[1].set_ylabel("Frequency")

plt.tight_layout()
plt.savefig(
    os.path.join(output_dir, "residual_analysis.png"), dpi=150, bbox_inches="tight"
)
plt.show()
plt.close()

print(f"Residuals Mean: {residuals.mean():.6f}")
print(f"Residuals Std: {residuals.std():.6f}")

In [ ]:
# Prepare test data with polynomial features
X_test = df_test_eng.drop(columns=["datetime"], errors="ignore")

# Apply polynomial transformation
poly_test = poly.transform(X_test[weather_cols])
poly_test_df = pd.DataFrame(
    poly_test, columns=poly.get_feature_names_out(weather_cols), index=X_test.index
)

X_test_poly = pd.concat([X_test.drop(columns=weather_cols), poly_test_df], axis=1)

# Scale test features using training scaler
X_test_poly_sc = scaler.transform(X_test_poly)

# Make predictions using best Lasso model
pred_test_log = lasso_final.predict(X_test_poly_sc)
pred_test = np.clip(np.expm1(pred_test_log), 0, None)

print(f"Test predictions shape: {pred_test.shape}")
print(
    f"Test predictions - Min: {pred_test.min():.2f}, Max: {pred_test.max():.2f}, Mean: {pred_test.mean():.2f}"
)
print(f"Test predictions - Sample values: {pred_test[:5]}")

In [ ]:
# Create submission dataframe
submission = pd.DataFrame(
    {"datetime": df_test["datetime"], "count_predicted": pred_test}
)

# Verify submission format
print("\n" + "=" * 60)
print("SUBMISSION FILE VALIDATION")
print("=" * 60)
print(f"Number of rows: {len(submission)} (Expected: {len(df_test)})")
print(f"Number of columns: {len(submission.columns)} (Expected: 2)")
print(f"Column names: {submission.columns.tolist()}")
print(f"\nMissing values: {submission.isna().sum().sum()}")
print(f"Any negative predictions: {(submission['count_predicted'] < 0).sum()}")

print(f"\nFirst 10 rows of submission:")
print(submission.head(10))

# Save submission
submission.to_csv("submission.csv", index=False)
print(f"\n✓ Submission file saved as 'submission.csv'")
print(f"✓ File size: {os.path.getsize('submission.csv') / 1024:.2f} KB")

In [ ]:
print("\n" + "=" * 80)
print("ASSIGNMENT SUMMARY")
print("=" * 80)

print("\n1. DATA OVERVIEW:")
print(f"   - Training samples: {len(df_train):,}")
print(f"   - Test samples: {len(df_test):,}")
print(
    f"   - Features engineered: {X_train_poly_sc.shape[1]} (after polynomial expansion)"
)

print("\n2. BEST MODEL PERFORMANCE:")
print(f"   - Model: Lasso Regression with Polynomial Features")
print(f"   - Best α (regularization): {best_lasso_alpha}")
print(f"   - Training RMSLE: {train_rmsle_lasso:.6f}")
print(f"   - Validation RMSLE: {best_lasso_val_rmsle:.6f}")
print(
    f"   - Improvement over baseline: {((val_rmsle_base - best_lasso_val_rmsle) / val_rmsle_base * 100):.2f}%"
)

print("\n3. KEY FEATURE ENGINEERING:")
print("   - Cyclical encoding for hour, month, day-of-week")
print("   - Peak hour indicators for commute patterns")
print("   - Weather interactions (temp*humidity, etc.)")
print("   - Polynomial weather features (degree=2)")
print("   - Log transformation of target variable")

print("\n4. SUBMISSION FILE:")
print(f"   - File: submission.csv")
print(f"   - Rows: {len(submission):,}")
print(f"   - Columns: {', '.join(submission.columns.tolist())}")
print(
    f"   - Prediction range: [{submission['count_predicted'].min():.2f}, {submission['count_predicted'].max():.2f}]"
)

print("\n" + "=" * 80)